# System 3: Manufacturing Assessment Pipeline

This notebook runs System 3 to assess whether a candidate material proposed by System 2 is manufacturable at lab scale.

**Prerequisite**: Run `01_system1_property_extraction.ipynb` and `02_system2_material_discovery.ipynb` first. Then load their outputs below (or use saved files from `pipeline_logs/`).

**Workflow**: Load S1/S2 outputs -> Process retrieval (ChromaDB) -> Feasibility assessment -> If manufacturable: process recipe; if blocked: blocking constraints + tracker update.

- **status = manufacturable**: process_recipe, evidence
- **status = blocked**: blocking_constraints, feedback_to_system2; rejection added to tracker (source=manufacturability)

## Setup: Import modules and load configuration

In [1]:
import sys
import os
import json
import glob
from pathlib import Path

# Set project_root to workspace root - navigate up from notebooks directory
# If running from notebook, try to find project root by looking for pipeline_logs
current_dir = Path().resolve()
if (current_dir / "src").exists() and (current_dir / "config").exists():
    project_root = current_dir
elif (current_dir.parent / "src").exists() and (current_dir.parent / "config").exists():
    project_root = current_dir.parent
else:
    # Fallback: check SYS3DEV_ROOT env var, otherwise assume parent of notebooks/
    import os
    project_root = Path(os.environ.get("SYS3DEV_ROOT", current_dir.parent))
sys.path.insert(0, str(project_root))
print(f"project_root set to: {project_root}")

from src.config import load_config
from src.utils import llm, TransformerEmbeddingFunction, ChatLogger
from src.agents import ResearchAnalyst, ResearchManager, RejectedCandidateTracker, MultiAnalyst
from src.pipelines import run_manufacturability_assessment_pipeline

from chromadb import PersistentClient
from sentence_transformers import SentenceTransformer

config = load_config()
print("Configuration loaded")

project_root set to: /orcd/pool/004/tphage/SG_feb/Sys3Dev
Configuration loaded


## Initialize LLM and embeddings

In [2]:
llm_config = {
    "api_key": config["llm"]["api_key"],
    "base_url": config["llm"]["base_url"],
    "model": config["llm"]["model_name"],
    "max_tokens": config["llm"]["max_tokens"]
}
llm_instance = llm(llm_config)
generate = llm_instance.generate_cli
print("LLM wrapper initialized")

embedding_tokenizer = ""
embedding_model = SentenceTransformer(config["embeddings"]["model_name"], trust_remote_code=True)
embedding_function = TransformerEmbeddingFunction(
    embedding_tokenizer=embedding_tokenizer,
    embedding_model=embedding_model
)
print("Embedding model initialized")

LLM wrapper initialized


<All keys matched successfully>


Embedding model initialized


## Load ChromaDB corpora for process retrieval

Load manufacturing_textbooks, spec_sheets, and optionally patents and materialdb. If a corpus is missing, that source is skipped (graceful degradation).

In [3]:
chroma_cfg = config["data"]["chromadb"]
base_path = chroma_cfg.get("base_path", "")
n_results_per_source = config.get("pipelines", {}).get("manufacturability_assessment", {}).get("n_results_per_source", 5)

analysts = {}

for source_key, cfg_key in [
    ("manufacturing_textbooks", "manufacturing_textbooks"),
    ("spec_sheets", "spec_sheets"),
    ("patents", "patents"),
    ("materialdb", "materialdb"),
]:
    try:
        source_cfg = chroma_cfg.get(cfg_key)
        if not source_cfg:
            continue
        db_path = os.path.join(base_path, source_cfg["database_path"]) if base_path else source_cfg["database_path"]
        if not os.path.exists(db_path):
            print(f"Skipping {source_key}: path not found {db_path}")
            continue
        client = PersistentClient(path=db_path)
        collections = client.list_collections()
        coll_name = source_cfg.get("collection_name") or (collections[0].name if collections else None)
        if not coll_name:
            continue
        collection = client.get_collection(coll_name, embedding_function=embedding_function)
        analysts[source_key] = ResearchAnalyst(
            collection=collection,
            embedding_function=embedding_function,
            n_results=n_results_per_source,
        )
        print(f"Loaded ChromaDB: {source_key}")
    except Exception as e:
        print(f"Skipping {source_key}: {e}")

if not analysts:
    print("Warning: No ChromaDB sources loaded. Pipeline will return blocked (missing critical info).")
process_analyst = MultiAnalyst(analysts) if analysts else None
print(f"MultiAnalyst has {len(analysts)} sources: {list(analysts.keys())}")

Loaded ChromaDB: manufacturing_textbooks
Skipping spec_sheets: path not found /orcd/home/002/tphage/orcd/pool/SG_Ph1_Data/ChromaDBs/SpecSheets/chromaDB_specsheets
Loaded ChromaDB: patents
Loaded ChromaDB: materialdb
MultiAnalyst has 3 sources: ['manufacturing_textbooks', 'patents', 'materialdb']


## Initialize ResearchManager and tracker

In [4]:
# Same naming as S1/S2: system3_chat_log_manufacturability_assessment_{run_id}.json (run_id set in next cell)
import uuid
chat_logger = ChatLogger(
    run_id=str(uuid.uuid4()),
    pipeline="manufacturability_assessment"
)
manager = ResearchManager(
    name="research_manager",
    system_message=None,
    generate_fn=generate,
    chat_logger=chat_logger,
)
tracker = RejectedCandidateTracker()
print("ResearchManager and tracker initialized")

ResearchManager and tracker initialized


## Load System 1 and System 2 outputs

Load from pipeline_logs (system1_*.json and optionally a saved system2 result). If you have just run notebook 02, you can set system2_result in memory and load system1 from the latest file.

In [5]:
# Load most recent System 1 output
system1_files = glob.glob(os.path.join(project_root, "pipeline_logs", "system1_*.json"))
if system1_files:
    system1_files.sort(key=os.path.getmtime, reverse=True)
    with open(system1_files[0], "r", encoding="utf-8") as f:
        system1_data = json.load(f)
    print(f"Loaded System 1 from {system1_files[0]}")
    # Extract required fields - raise KeyError if missing (no defaults)
    try:
        properties_W = system1_data["properties_W"]
        material_X = system1_data["material_X"]
        application_Y = system1_data["application_Y"]
        initial_query = system1_data["sentence"]
        constraints_U = system1_data.get("constraints_U", [])
        if not constraints_U and "constraints" in system1_data:
            constraints_U = system1_data["constraints"]
    except KeyError as e:
        raise KeyError(
            f"Missing required field '{e.args[0]}' in System 1 JSON file. "
            f"Available keys: {list(system1_data.keys())}"
        ) from e
else:
    raise FileNotFoundError(
        "No System 1 output files found in pipeline_logs/. "
        "Please run notebook 01_system1_property_extraction.ipynb first to generate System 1 results."
    )

# System 2 result: load from most recent System 2 output file
system2_files = glob.glob(os.path.join(project_root, "pipeline_logs", "system2_*.json"))
if system2_files:
    # Sort by modification time, get most recent
    system2_files.sort(key=os.path.getmtime, reverse=True)
    system2_result_path = system2_files[0]
    with open(system2_result_path, "r", encoding="utf-8") as f:
        system2_result = json.load(f)
    print(f"Loaded System 2 from {system2_result_path}")
else:
    raise FileNotFoundError(
        "No System 2 output files found in pipeline_logs/. "
        "Please run notebook 02_system2_material_discovery.ipynb first to generate System 2 results."
    )

# Align System 3 run_id with S1/S2: system3_{base_run_id}_{counter}.json
if system1_data and "chat_logger" in globals():
    system1_run_id = system1_data.get("run_id")
    base_run_id = system1_run_id.split("_")[0] if (system1_run_id and "_" in system1_run_id) else (system1_run_id or "")
    if not base_run_id:
        from datetime import datetime
        base_run_id = datetime.now().strftime("%Y%m%d%H")
    output_dir = os.path.join(project_root, "pipeline_logs")
    pattern = os.path.join(output_dir, "system3_" + base_run_id + "_*.json")
    existing = glob.glob(pattern)
    max_counter = -1
    for f in existing:
        try:
            n = int(os.path.basename(f).replace(".json", "").split("_")[-1])
            max_counter = max(max_counter, n)
        except (ValueError, IndexError):
            pass
    chat_logger.run_id = base_run_id + "_" + str(max_counter + 1)
    print("System 3 run_id:", chat_logger.run_id)

print(f"material_X={material_X}, application_Y={application_Y}, properties_W keys={list(properties_W.keys())}")

Loaded System 1 from /orcd/pool/004/tphage/SG_feb/Sys3Dev/pipeline_logs/system1_2026021912_0.json
Loaded System 2 from /orcd/pool/004/tphage/SG_feb/Sys3Dev/pipeline_logs/system2_2026021912_0.json
System 3 run_id: 2026021912_0
material_X=PTFE, application_Y=industrial seals, properties_W keys=['required', 'target_values']


## Run Manufacturing Assessment pipeline

In [6]:
result = run_manufacturability_assessment_pipeline(
    system2_result=system2_result,
    initial_query=initial_query,
    material_X=material_X,
    application_Y=application_Y,
    constraints_U=constraints_U,
    tracker=tracker,
    process_analyst=process_analyst,
    manager=manager,
    properties_W=properties_W,
    config=config,
    temperature=0,
    chat_logger=chat_logger,
)

INFO:src.pipelines.manufacturability_assessment:System 3: Manufacturability assessment starting.


Manufacturability Assessment Pipeline: Starting Process

Pipeline Configuration:
  [OK] Candidate material: BASF PBI‑ELAST 2000
  [OK] Application: industrial seals
  [OK] Constraints: 0 constraint(s)
  [OK] Max process families: 10
  [OK] Temperature: 0
  [OK] Required properties: 32 property(ies)
  [OK] Tracker: Available

----------------------------------------------------------------------
Step 1: Process Retrieval
----------------------------------------------------------------------
→ Generating process retrieval queries...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

...

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

INFO:src.pipelines.manufacturability_assessment:System 3: Process retrieval collected 10 evidence items.


      ✓ Retrieved 15 documents from RAG

→ Deduplicating results...
  → Total documents before deduplication: 60
  → Unique documents after deduplication: 10
  → Removed 50 duplicate(s)
  → Capped at max_process_families=10
✓ Collected 10 evidence items

----------------------------------------------------------------------
Step 2: Feasibility Assessment
----------------------------------------------------------------------
→ Assessing manufacturability feasibility...
  → Candidate: BASF PBI‑ELAST 2000
  → Application: industrial seals
  → Reviewing 10 evidence items
  → Required properties: 32 property(ies)


INFO:httpx:HTTP Request: POST http://localhost:8081/v1/chat/completions "HTTP/1.1 200 OK"
INFO:src.pipelines.manufacturability_assessment:System 3: Added rejection to tracker (source=manufacturability).
INFO:src.pipelines.manufacturability_assessment:System 3: Returning blocked with 1 constraints.



  Feasibility Assessment:
     Feasible: [NO]

----------------------------------------------------------------------
Step 3: Blocking Constraints
----------------------------------------------------------------------
→ Processing blocking constraints...
✓ Identified 1 blocking constraint(s):
  1. [missing_critical_info] No documented evidence of a lab‑scale synthesis or compounding route f...

  Feedback to System 2: Exclude candidates for which no explicit synthesis route, precursor availability, or processing deta...

✓ Added rejection to tracker (source=manufacturability)

✓ Saved chat log to ./pipeline_logs/chats/system3_chat_log_manufacturability_assessment_2026021912_0.json

Manufacturability Assessment: Process Complete
Summary:
  - Status: BLOCKED
  - Evidence items retrieved: 10
  - Blocking constraints: 1


In [7]:
if result.get("chat_log_path"):
    print("Chat log saved:", result["chat_log_path"])
output_dir = os.path.join(project_root, "pipeline_logs")
run_id = chat_logger.run_id if "chat_logger" in globals() and hasattr(chat_logger, "run_id") else "unknown"
output_path = os.path.join(output_dir, "system3_" + run_id + ".json")
with open(output_path, "w", encoding="utf-8") as f:
    result["run_id"] = run_id
    json.dump(result, f, indent=2, default=str)
print("Result saved to", output_path)
print("Manufacturing assessment complete")

Chat log saved: ./pipeline_logs/chats/system3_chat_log_manufacturability_assessment_2026021912_0.json
Result saved to /orcd/pool/004/tphage/SG_feb/Sys3Dev/pipeline_logs/system3_2026021912_0.json
Manufacturing assessment complete


## Inspect result

In [8]:
status = result.get("status", "")
print(f"Status: {status}")
print(f"Manufacturable (legacy): {result.get('manufacturable')}")
if status == "manufacturable":
    print("Process recipe:")
    for step in result.get("process_recipe", []):
        print(f"  Step {step.get('step_index')}: {step.get('description')}")
        if step.get("conditions"):
            print(f"    Conditions: {step['conditions']}")
        if step.get("equipment_class"):
            print(f"    Equipment: {step['equipment_class']}")
    print(f"Evidence: {result.get('evidence')}")
elif status == "blocked":
    print("Blocking constraints:")
    for c in result.get("blocking_constraints", []):
        print(f"  [{c.get('type')}] {c.get('description')}")
    print(f"Feedback to System 2: {result.get('feedback_to_system2')}")
    print(f"Rejected candidate (legacy): {result.get('rejected_candidate')}")

Status: blocked
Manufacturable (legacy): False
Blocking constraints:
  [missing_critical_info] No documented evidence of a lab‑scale synthesis or compounding route for BASF PBI‑ELAST 2000, including monomer availability, polymerization conditions, or processing parameters. The retrieved patents and textbooks discuss silicone and generic thermoplastic elastomers but do not provide the specific chemistry or manufacturing steps required for this fluoropolymer.
Feedback to System 2: Exclude candidates for which no explicit synthesis route, precursor availability, or processing details are documented; prioritize materials with publicly available manufacturing data or clear patent disclosures.
Rejected candidate (legacy): {'candidate': 'BASF\u202fPBI‑ELAST\u202f2000', 'constraints': ['No documented evidence of a lab‑scale synthesis or compounding route for BASF\u202fPBI‑ELAST\u202f2000, including monomer availability, polymerization conditions, or processing parameters. The retrieved patents